# 11 Treatment assignment check

**Question.** Our main results assume that every user had the same chance of being treated (85 percent). Is that true in this file, and does it change the conclusions?

**Idea.** If treatment was assigned purely at random, nobody can predict who was treated from the features, and a model that tries should score an AUC of 0.50. If it scores clearly above 0.50, assignment depended a little on the features. We then weight each user by the inverse of their estimated chance of the treatment they got, and repeat the average effect and the Qini coefficients.

**Care taken.** Propensity models are trained on development rows only (the first 1M rows of the shuffled training pool). They are applied to the holdout, which they never saw. A placebo run, with treatment shuffled before training, shows the method does not move the average effect by itself.

**Inputs:** `data/criteo.parquet`, `results/scores_5m.parquet`

**Outputs:** `results/propensity_summary.csv`, `results/propensity_weighted_qini.csv`, `results/treated_share_by_score_decile.csv`

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().parent
sys.path.append(str(ROOT / "src"))
from uplift.data import FEATURES, make_splits, take
from uplift.metrics import qini_coefficient

results_dir = ROOT / "results"
df = pd.read_parquet(ROOT / "data" / "criteo.parquet", columns=FEATURES + ["treatment"])
test_ids, pool_ids = make_splits(len(df))
dev = take(df, pool_ids[:1_000_000])

scores = pd.read_parquet(results_dir / "scores_5m.parquet")
assert np.array_equal(np.sort(scores.index.to_numpy()), np.sort(test_ids)), "scores file is not the frozen holdout"
hold_X = df.iloc[scores.index.to_numpy()][FEATURES]
y = scores["visit"].to_numpy(dtype=float)
t = scores["treatment"].to_numpy(dtype=float)
print(f"Development rows: {len(dev):,}  Holdout rows: {len(hold_X):,}")

Development rows: 1,000,000  Holdout rows: 2,000,000


## 1. Treated share by response score decile

Users are ordered by the response baseline score on the holdout. If assignment were independent of the features, every decile would hold about 85 percent treated users.

In [2]:
rank = scores["response_visit"].rank(method="first", ascending=False)
decile = pd.qcut(rank, 10, labels=range(1, 11))
share = scores.groupby(decile, observed=True)["treatment"].mean().rename("treated_share").reset_index().rename(columns={"response_visit": "decile"})
share.columns = ["decile", "treated_share"]
share.to_csv(results_dir / "treated_share_by_score_decile.csv", index=False)
display(share.round(4).T)

,0,1,2,3,4,5,6,7,8,9
decile,1.00,2.0000,3.0000,4.0000,5.0000,6.0000,7.0000,8.000,9.0000,10.0000
treated_share,0.86,0.8526,0.8493,0.8458,0.8464,0.8486,0.8481,0.849,0.8489,0.8479


## 2. Can a model predict who was treated, and how much does weighting change the average effect?

Three propensity models, plus a placebo. Chances are clipped to 0.05 to 0.95 so that no user gets a huge weight. The weighted average effect uses normalized weights.

In [3]:
def weighted_ate(propensity, clip=(0.05, 0.95)):
    e = np.clip(propensity, *clip)
    w = t / e + (1 - t) / (1 - e)
    treated = (w * t * y).sum() / (w * t).sum()
    control = (w * (1 - t) * y).sum() / (w * (1 - t)).sum()
    return treated - control

candidates = {
    "LightGBM, 200 trees, 31 leaves": LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=31, min_child_samples=200, verbose=-1, random_state=0),
    "LightGBM, 50 trees, 7 leaves": LGBMClassifier(n_estimators=50, learning_rate=0.05, num_leaves=7, min_child_samples=2000, verbose=-1, random_state=0),
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=300)),
}
unweighted = y[t == 1].mean() - y[t == 0].mean()
rows = [{"propensity_model": "none (unweighted)", "holdout_auc": np.nan, "chance_min": np.nan, "chance_max": np.nan, "ate_visit": unweighted}]
fitted = {}
for name, model in candidates.items():
    model.fit(dev[FEATURES], dev["treatment"])
    e = model.predict_proba(hold_X)[:, 1]
    fitted[name] = e
    rows.append({"propensity_model": name, "holdout_auc": roc_auc_score(t, e), "chance_min": e.min(), "chance_max": e.max(), "ate_visit": weighted_ate(e)})

shuffled = dev.copy()
shuffled["treatment"] = np.random.default_rng(0).permutation(shuffled["treatment"].to_numpy())
placebo = LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=31, min_child_samples=200, verbose=-1, random_state=0).fit(shuffled[FEATURES], shuffled["treatment"])
e_placebo = placebo.predict_proba(hold_X)[:, 1]
rows.append({"propensity_model": "Placebo (treatment shuffled before training)", "holdout_auc": roc_auc_score(t, e_placebo), "chance_min": e_placebo.min(), "chance_max": e_placebo.max(), "ate_visit": weighted_ate(e_placebo)})

summary = pd.DataFrame(rows)
summary.to_csv(results_dir / "propensity_summary.csv", index=False)
display(summary.round(5))

,propensity_model,holdout_auc,chance_min,chance_max,ate_visit
0,none (unweighted),NaN,NaN,NaN,0.00995
1,"LightGBM, 200 trees, 31 leaves",0.50801,0.71779,0.98217,0.00669
2,"LightGBM, 50 trees, 7 leaves",0.50914,0.84221,0.92041,0.00752
3,Logistic regression,0.50866,0.81857,0.94121,0.00709
4,Placebo (treatment shuffled before training),0.50077,0.70145,0.94480,0.00974


## 3. Do the Qini coefficients change?

Same Qini formula as before, with weighted rates inside each top share of users. The propensity model is the first LightGBM one.

In [4]:
def weighted_qini_coefficient(score, propensity, clip=(0.05, 0.95)):
    e = np.clip(propensity, *clip)
    w = t / e + (1 - t) / (1 - e)
    order = np.argsort(-score, kind="stable")
    yo, to, wo = y[order], t[order], w[order]
    wt, wc = wo * to, wo * (1 - to)
    cum_t, cum_c = np.cumsum(wt), np.cumsum(wc)
    rate_t = np.divide(np.cumsum(wt * yo), cum_t, out=np.zeros_like(cum_t), where=cum_t > 0)
    rate_c = np.divide(np.cumsum(wc * yo), cum_c, out=np.zeros_like(cum_c), where=cum_c > 0)
    n = len(yo)
    q = np.arange(1, n + 1) * (rate_t - rate_c) / n
    q[(cum_t == 0) | (cum_c == 0)] = 0.0
    random_line = np.arange(1, n + 1) / n * q[-1]
    return float((q - random_line).mean())

e_main = fitted["LightGBM, 200 trees, 31 leaves"]
qrows = []
for model in ["response", "s", "t", "x"]:
    s = scores[f"{model}_visit"].to_numpy()
    qrows.append({"model": model, "qini_unweighted": qini_coefficient(y, t, s), "qini_weighted": weighted_qini_coefficient(s, e_main)})
qtable = pd.DataFrame(qrows)
qtable["gap_unweighted"] = qtable["qini_unweighted"] - qtable.loc[0, "qini_unweighted"]
qtable["gap_weighted"] = qtable["qini_weighted"] - qtable.loc[0, "qini_weighted"]
qtable.to_csv(results_dir / "propensity_weighted_qini.csv", index=False)
display(qtable.round(6))

,model,qini_unweighted,qini_weighted,gap_unweighted,gap_weighted
0,response,0.003402,0.002912,0.000000,0.000000
1,s,0.003559,0.003064,0.000157,0.000152
2,t,0.003579,0.002972,0.000177,0.000060
3,x,0.003687,0.003124,0.000285,0.000212


In [5]:
models = summary.iloc[1:4]
placebo_row = summary.iloc[4]
uplift_rows = qtable.iloc[1:]
still_beat = bool((uplift_rows["gap_weighted"] > 0).all())
best_weighted = qtable.loc[qtable["qini_weighted"].idxmax(), "model"]
order_same = list(qtable.sort_values("qini_unweighted")["model"]) == list(qtable.sort_values("qini_weighted")["model"])

display(Markdown(
    f"A model predicts who was treated with a holdout AUC of {models['holdout_auc'].min():.3f} to {models['holdout_auc'].max():.3f}, "
    f"against {placebo_row['holdout_auc']:.3f} for the placebo, so assignment was not perfectly independent of the features. "
    f"The treated share is {share.loc[0, 'treated_share']:.1%} in the top tenth by response score and "
    f"{share.loc[1:, 'treated_share'].mean():.1%} in the other nine tenths. "
    f"Weighting lowers the average visit effect from {unweighted:.4f} to between {models['ate_visit'].min():.4f} and {models['ate_visit'].max():.4f}, "
    f"while the placebo gives {placebo_row['ate_visit']:.4f}. "
    f"With weights, {'every' if still_beat else 'not every'} uplift model still beats the response baseline, the best model is {best_weighted}, "
    f"and the ranking of the four models is {'unchanged' if order_same else 'changed'}."
))

A model predicts who was treated with a holdout AUC of 0.508 to 0.509, against 0.501 for the placebo, so assignment was not perfectly independent of the features. The treated share is 86.0% in the top tenth by response score and 84.9% in the other nine tenths. Weighting lowers the average visit effect from 0.0100 to between 0.0067 and 0.0075, while the placebo gives 0.0097. With weights, every uplift model still beats the response baseline, the best model is x, and the ranking of the four models is changed.